# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tushar-sharma001/Flyrank-Ml-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import HfApi

TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

MONTH = "2026-03"
BASE = "hf://datasets/FlyRank/internship-warehouse"

api = HfApi()
all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=TOKEN)
fact_files = [f for f in all_files if "fact_content_daily_performance" in f and f"month={MONTH}" in f]
next_month_files = [f for f in all_files if "fact_content_daily_performance" in f and "month=2026-04" in f]
dim_content_files = [f for f in all_files if "dim_content" in f.lower() and f.endswith(".parquet")]

FACT = [f"{BASE}/{f}" for f in fact_files]
FACT_NEXT = [f"{BASE}/{f}" for f in next_month_files]
DIM_CONTENT = [f"{BASE}/{f}" for f in dim_content_files]

FACT_STR = "[" + ", ".join(f"'{p}'" for p in FACT) + "]"
FACT_TWO_MONTHS_STR = "[" + ", ".join(f"'{p}'" for p in FACT + FACT_NEXT) + "]"
DIM_CONTENT_STR = "[" + ", ".join(f"'{p}'" for p in DIM_CONTENT) + "]"

print("fact files:", len(fact_files), "| dim_content files:", len(dim_content_files))

fact files: 1 | dim_content files: 1


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building out a fuller feature vector than w03's quick 3-feature version — this
one's meant to actually cover what I'd realistically train on, categoricals
included, with a real plan for missing values instead of just dropping rows.

In [3]:
# grabbing content metadata first, this is the stuff that's basically static per page
content = con.sql(f"SELECT content_hash_id, content_created_date FROM read_parquet({DIM_CONTENT_STR})").df()

# daily facts for march -- this is where most of the actual signal lives
daily = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date,
           gsc_impressions, gsc_clicks, gsc_avg_position,
           sessions_direct, sessions_organic, sessions_paid, sessions_social, sessions_ai,
           ga4_data_available
    FROM read_parquet({FACT_STR})
""").df()
daily["report_date"] = pd.to_datetime(daily["report_date"])

# rolling everything up to one row per content item -- this is my actual grain going forward
feat = daily.groupby(["content_hash_id", "client_hash_id"]).agg(
    impressions_90d=("gsc_impressions", "sum"),
    clicks_90d=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    days_with_impressions=("gsc_impressions", lambda x: (x > 0).sum()),
).reset_index()

# ctr is derived, not raw -- computing it here instead of trusting some upstream column
feat["ctr_90d"] = feat["clicks_90d"] / feat["impressions_90d"].replace(0, np.nan)

# ga4 sessions, but only counted where the flag says the data's actually real
ga4_sessions = daily[daily["ga4_data_available"] == True].groupby(["content_hash_id", "client_hash_id"]).agg(
    sessions_90d=("sessions_organic", lambda x: x.fillna(0).sum())  # keeping it simple, just organic for now
).reset_index()
feat = feat.merge(ga4_sessions, on=["content_hash_id", "client_hash_id"], how="left")
# no ga4 data at all for this content? that's a real "we don't know", not a zero -- keeping it NaN on purpose

# bolt on the content metadata and turn age into something usable
feat = feat.merge(content, on="content_hash_id", how="left")
feat["content_created_date"] = pd.to_datetime(feat["content_created_date"])
feat["content_age_days"] = (pd.Timestamp("2026-03-31") - feat["content_created_date"]).dt.days

# categorical: bucket position into tiers instead of leaving it purely numeric,
# gives the model (and me) a more interpretable read
def position_tier(p):
    if pd.isna(p): return "unknown"
    if p <= 3: return "top3"
    if p <= 10: return "page1"
    if p <= 20: return "striking_distance"
    return "beyond"
feat["position_tier"] = feat["avg_position"].apply(position_tier)

print(f"rows: {len(feat)}, columns: {feat.shape[1]}")
print("\nmissing value check:")
print(feat.isna().sum())
feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 331437, columns: 11

missing value check:
content_hash_id               0
client_hash_id                0
impressions_90d               0
clicks_90d                    0
avg_position             154699
days_with_impressions         0
ctr_90d                  154699
sessions_90d             240948
content_created_date          0
content_age_days              0
position_tier                 0
dtype: int64


,content_hash_id,client_hash_id,impressions_90d,clicks_90d,avg_position,days_with_impressions,ctr_90d,sessions_90d,content_created_date,content_age_days,position_tier
0,content_000005d4ced12088,client_9958f0a7ae1df715,86,0,72.854861,24,0.0,<NA>,2025-03-28,368,beyond
1,content_00001e488b74b799,client_625b6439094e23e4,0,0,NaN,0,NaN,<NA>,2025-04-18,347,unknown
2,content_00007bd2985b77c3,client_73cda7b4e4f265ea,47,0,5.269565,23,0.0,<NA>,2025-07-31,243,page1
3,content_00008950670cb6b5,client_def0955f7a377868,0,0,NaN,0,NaN,0,2025-07-11,263,unknown
4,content_0000a348850eb1fc,client_3ffa76342f366962,0,0,NaN,0,NaN,0,2025-09-02,210,unknown


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

One line per feature: what it means, how I'm handling missing, whether it's
categorical, and — the important one — was it actually knowable before the
decision point.

In [4]:
feature_notes = pd.DataFrame([
    {"feature": "impressions_90d", "meaning": "total GSC impressions, summed over the month",
     "missing_handling": "0 if no impressions rows, never NaN", "categorical": False, "available_before_decision": True},
    {"feature": "clicks_90d", "meaning": "total GSC clicks",
     "missing_handling": "0 if no clicks, never NaN", "categorical": False, "available_before_decision": True},
    {"feature": "avg_position", "meaning": "mean search position across the month",
     "missing_handling": "NaN if zero impressions (position is meaningless with no visibility) -- left as NaN, not imputed to a fake number", "categorical": False, "available_before_decision": True},
    {"feature": "ctr_90d", "meaning": "clicks / impressions, computed here not trusted from upstream",
     "missing_handling": "NaN when impressions_90d is 0 -- dividing by zero shouldn't become a fake 0% ctr", "categorical": False, "available_before_decision": True},
    {"feature": "sessions_90d", "meaning": "organic GA4 sessions, only counted where ga4_data_available is TRUE",
     "missing_handling": "left NaN when no real GA4 tracking exists yet -- NOT filled with 0, since that would claim 'zero engagement' when the truth is 'we don't know'", "categorical": False, "available_before_decision": True},
    {"feature": "content_age_days", "meaning": "days between content_created_date and the decision date",
     "missing_handling": "NaN if content_created_date itself is missing (rare, but happens)", "categorical": False, "available_before_decision": True},
    {"feature": "position_tier", "meaning": "bucketed version of avg_position -- top3 / page1 / striking_distance / beyond / unknown",
     "missing_handling": "explicit 'unknown' bucket instead of dropping the row", "categorical": True, "available_before_decision": True},
])
feature_notes

,feature,meaning,missing_handling,categorical,available_before_decision
0,impressions_90d,"total GSC impressions, summed over the month","0 if no impressions rows, never NaN",False,True
1,clicks_90d,total GSC clicks,"0 if no clicks, never NaN",False,True
2,avg_position,mean search position across the month,NaN if zero impressions (position is meaningle...,False,True
3,ctr_90d,"clicks / impressions, computed here not truste...",NaN when impressions_90d is 0 -- dividing by z...,False,True
4,sessions_90d,"organic GA4 sessions, only counted where ga4_d...",left NaN when no real GA4 tracking exists yet ...,False,True
5,content_age_days,days between content_created_date and the deci...,NaN if content_created_date itself is missing ...,False,True
6,position_tier,bucketed version of avg_position -- top3 / pag...,explicit 'unknown' bucket instead of dropping ...,True,True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Actually attacking my own feature set now -- not just checking the usual
suspects, but testing each candidate against a real label the way the
training-honest-models skill wants.

In [5]:
# building the same future_decline_label from w05/w06 so there's something real to test against
labeled = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, gsc_impressions,
           LEAD(gsc_impressions, 30) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS impressions_plus30
    FROM read_parquet({FACT_TWO_MONTHS_STR})
""").df()
labeled = labeled[labeled["report_date"] < "2026-04-01"].dropna(subset=["impressions_plus30"])
labeled["future_decline_label"] = (labeled["impressions_plus30"] < labeled["gsc_impressions"]).astype(int)
label_by_content = labeled.groupby("content_hash_id")["future_decline_label"].max()

check = feat.merge(label_by_content, on="content_hash_id", how="inner")

# test 1: does any single feature correlate suspiciously close to 1 with the label?
# that's the "too good to be true" signal that usually means leakage
numeric_feats = ["impressions_90d", "clicks_90d", "avg_position", "ctr_90d", "sessions_90d", "content_age_days"]
corrs = check[numeric_feats].corrwith(check["future_decline_label"]).sort_values(key=abs, ascending=False)
print("correlation of each feature with the label:")
print(corrs)
print("\nanything above ~0.9 here would be a real red flag -- nothing should be that clean")

# test 2: explicit check that nothing in my feature list is literally the label or its ingredients
banned = ["health_score", "priority_score", "action_type", "future_decline_label",
          "impressions_plus30", "trend_direction", "trend_pct"]
used_banned = [c for c in feat.columns if c in banned]
print("\nbanned/label-derived columns actually present in my feature vector:", used_banned, "(should be empty)")

# test 3: window overlap check -- none of my features should be summed over a window
# that overlaps march 31 into the future window used for the label
print("\nfeature window: all of March, up to 2026-03-31")
print("label window: the 30 days AFTER each row's report_date, extending into April")
print("no overlap by construction -- features never look past their own report_date")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

correlation of each feature with the label:
impressions_90d     0.194078
content_age_days   -0.183579
clicks_90d          0.113458
sessions_90d        0.103215
avg_position        0.028380
ctr_90d            -0.009812
dtype: float64

anything above ~0.9 here would be a real red flag -- nothing should be that clean

banned/label-derived columns actually present in my feature vector: [] (should be empty)

feature window: all of March, up to 2026-03-31
label window: the 30 days AFTER each row's report_date, extending into April
no overlap by construction -- features never look past their own report_date


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

The stuff I looked at and deliberately didn't use, and why.

In [6]:
excluded = pd.DataFrame([
    {"field": "health_score / priority_score / action_type", "why_excluded": "product-decision flags, not shipped in this dataset on purpose -- using them would mean copying FlyRank's existing rule instead of finding real signal"},
    {"field": "trend_direction / trend_pct", "why_excluded": "these ARE how I built the label -- using them as a feature too would be training the model to predict its own ingredients"},
    {"field": "raw query / url / title text", "why_excluded": "not shipped in the release at all -- scrambled before I ever see it, and I wouldn't use it even if I could (privacy)"},
    {"field": "sessions_paid / sessions_social / sessions_ai individually", "why_excluded": "rolled these into one sessions_90d for now rather than treating 5 separate sparse columns as 5 separate features -- ai sessions especially are too sparse on their own per the lane guide's own warning (30K rows out of 78M)"},
    {"field": "keyword_hash_id / url_hash_id", "why_excluded": "join/grouping keys only -- the codes themselves carry no real signal, using them as features would just be memorizing IDs"},
])
excluded

,field,why_excluded
0,health_score / priority_score / action_type,"product-decision flags, not shipped in this da..."
1,trend_direction / trend_pct,these ARE how I built the label -- using them ...
2,raw query / url / title text,not shipped in the release at all -- scrambled...
3,sessions_paid / sessions_social / sessions_ai ...,rolled these into one sessions_90d for now rat...
4,keyword_hash_id / url_hash_id,join/grouping keys only -- the codes themselve...


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.